Log-rank tests

Now formally test whether those group-wise survival differences are significant.

This matters because it helps you:

- identify variables associated with survival differences
- support your conclusions with statistics, not just visuals
- decide which features are promising for deeper analysis

-----------------------------------
Log-rank tests work best for grouped/categorical variables, such as:

anaemia
diabetes
high_blood_pressure
sex
smoking

You can also use it for continuous variables, but only after grouping/binning them, for example:

age: younger vs older
serum_creatinine: low vs high
ejection_fraction: low/medium/high

----------------------------------------
How to interpret the result

Suppose you compare survival for:

diabetes = 0
diabetes = 1

The log-rank test gives a p-value.

If p < 0.05

There is evidence that the survival distributions differ between the groups.

If p >= 0.05

There is not enough evidence to say the survival curves differ significantly.

Important:
This does not tell you the size of the effect or direction in a multivariable sense. It only tells you whether the survival curves differ.

In [10]:
from lifelines.statistics import logrank_test

In [11]:
import pandas as pd

data_path = "../data/Heart_failure_clinical_records_dataset.csv"  
df = pd.read_csv(data_path)
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [14]:
#example of logrank test for diabetes vs non-diabetes

group0 = df[df['diabetes'] == 0]
group1 = df[df['diabetes'] == 1]

results = logrank_test(
    durations_A = group0['time'], 
    durations_B = group1['time'], 
    event_observed_A = group0['DEATH_EVENT'], 
    event_observed_B = group1['DEATH_EVENT']
    )

print("test statistic: ", results.test_statistic)
print("p-value: ", results.p_value)

test statistic:  0.04052787802904746
p-value:  0.840451985394602


In [19]:
# run for all binary variables

binary_vars = ['anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking']

logrank_results = []

for var in binary_vars:
    group_0 = df[df[var] == 0]
    group_1 = df[df[var] == 1]

    result = logrank_test(
        durations_A=group_0['time'],
        durations_B=group_1['time'],
        event_observed_A=group_0['DEATH_EVENT'],
        event_observed_B=group_1['DEATH_EVENT']
    )

    logrank_results.append({
        'variable': var,
        'test_statistic': result.test_statistic,
        'p_value': result.p_value
    })

import pandas as pd
logrank_df = pd.DataFrame(logrank_results).sort_values('p_value')
# print(logrank_df)

# add a column to indicate significance for p < 0.05
logrank_df['significant'] = logrank_df['p_value'] < 0.05
print(logrank_df)

              variable  test_statistic   p_value  significant
2  high_blood_pressure        4.406248  0.035808         True
0              anaemia        2.726464  0.098698        False
1             diabetes        0.040528  0.840452        False
3                  sex        0.003971  0.949752        False
4              smoking        0.002042  0.963960        False


---------------
After plotting univariate Kaplan–Meier survival curves for grouped variables, log-rank tests were conducted to determine whether survival differences between groups were statistically significant. This step helped identify categorical predictors potentially associated with survival outcomes and informed variable selection for subsequent Cox proportional hazards modeling.
----------------